# Exercise 3.4: RDataFrame versus the explicit loop revised

From *Programming in High Energy Particle Physics*, Chapter 3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch03/ex03_4.ipynb)

Exercise 3.4 RDataFrame versus the explicit loop revised Implement the dilepton-mass analysis of Exercise 3.1 twice: once with an explicit event loop as in Listing 3.7 , once with RDataFrame. Compare the code length and readability. Benchmark the execution time with and without ROOT::EnableImplicitMT() on a tree with 100,000+ events. Report the speedup factor.

<details><summary>Hint</summary>

Use TStopwatch around the first access to a result, and repeat each measurement a few times because the first run also includes JIT compilation and cold file caches. A tree of 100,000 small events may be too small to benefit from many threads: try increasing the number of events in Listing 3.6 to a few million.

</details>

**Colab setup:** ROOT installation takes about 3–5 minutes and restarts the runtime once. After that restart, run all cells from the top again. The setup cells are safe to rerun.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        import ROOT
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q condacolab")
        import condacolab
        condacolab.install()  # one-time runtime restart

In [ ]:
if IN_COLAB:
    try:
        import ROOT
    except ImportError:
        get_ipython().system("mamba install -q -y -c conda-forge root")
        import ROOT
else:
    import ROOT
ROOT.gROOT.SetBatch(True)

### Generate and benchmark

`N_EVENTS=8000` keeps a first run short. Set it to at least 100,000 for the requested full benchmark; report both sizes. Use repeated timings.

In [ ]:
import math
from array import array
from pathlib import Path
def create_tree(n_events=8000):
    rng=ROOT.TRandom3(42)
    f=ROOT.TFile("events.root","RECREATE")
    tree=ROOT.TTree("events","Toy dilepton events")
    floats={name:ROOT.std.vector('float')() for name in
        ('electron_pt','electron_eta','electron_phi','electron_iso',
         'muon_pt','muon_eta','muon_phi','muon_iso')}
    ints={name:ROOT.std.vector('int')() for name in ('electron_charge','muon_charge')}
    has_Z=array('i',[0]);z_pass=array('i',[0]);y_Z=array('f',[0.])
    for name,v in {**floats,**ints}.items():tree.Branch(name,v)
    tree.Branch('has_Z',has_Z,'has_Z/I');tree.Branch('z_pass',z_pass,'z_pass/I')
    tree.Branch('y_Z',y_Z,'y_Z/F')
    for _ in range(n_events):
        for v in [*floats.values(),*ints.values()]:v.clear()
        has_Z[0]=int(rng.Rndm()<0.35);z_pass[0]=0
        y_Z[0]=rng.Gaus(0,1.3) if has_Z[0] else 0.0
        for flavor in ('electron','muon'):
            for _ in range(rng.Integer(3)):
                pt=rng.Exp(25)+5;eta=rng.Gaus(0,1.8)
                floats[flavor+'_pt'].push_back(pt)
                floats[flavor+'_eta'].push_back(eta)
                floats[flavor+'_phi'].push_back(rng.Uniform(-math.pi,math.pi))
                floats[flavor+'_iso'].push_back(rng.Exp(0.25))
                ints[flavor+'_charge'].push_back(1 if rng.Rndm()<0.5 else -1)
        if has_Z[0]:
            flavor='electron' if rng.Rndm()<0.5 else 'muon'
            mass=max(70.,rng.Gaus(91.1876,2.8))
            costh=rng.Uniform(-0.98,0.98);eta_star=math.atanh(costh)
            pt_true=0.5*mass*math.sqrt(1-costh*costh)
            z_pass[0]=int(pt_true>20 and abs(y_Z[0]+eta_star)<2.4
                          and abs(y_Z[0]-eta_star)<2.4)
            phi=rng.Uniform(-math.pi,math.pi)
            for charge,eta,azimuth in ((1,y_Z[0]+eta_star,phi),
                                       (-1,y_Z[0]-eta_star,phi+math.pi)):
                pt=pt_true*rng.Gaus(1,0.02)
                floats[flavor+'_pt'].push_back(pt)
                floats[flavor+'_eta'].push_back(eta)
                floats[flavor+'_phi'].push_back(azimuth)
                floats[flavor+'_iso'].push_back(rng.Exp(0.12/(1+pt/50)))
                ints[flavor+'_charge'].push_back(charge)
        tree.Fill()
    tree.Write();f.Close()
    print(f"Created {n_events} toy events in events.root")
N_EVENTS=8000  # set to 50_000 for the full exercise
create_tree(N_EVENTS)

In [ ]:
# TODO: implement explicit loop and RDataFrame pair-mass histograms.
# TODO: compare serial and ROOT.EnableImplicitMT(2) after repeated runs.

In [ ]:
if "n1" in globals() and "n3" in globals():
    assert n1 == n3
else:
    print("Complete both analyses, then compare pair counts.")